# 🧠 AI Stock Prediction — Training Notebook

**How to use this notebook:**
1. `Runtime → Change runtime type → Hardware accelerator → T4 GPU`
2. Run all cells top to bottom (`Runtime → Run all`)
3. Download `model_v2.pth`, `scaler_v2.pkl`, `model_v2_config.pth` at the end
4. Put those 3 files in your project folder alongside the Python scripts

**What this notebook does:**
- Installs dependencies
- Lets you upload a CSV of OHLCV data (or use yfinance to download it)
- Computes all 27 technical features
- Trains the Transformer model on the T4 GPU (free, ~15 min for 5 years)
- Downloads the trained model files back to your computer

## Step 1 — Check GPU

In [ ]:
import torch

print(f'PyTorch version:  {torch.__version__}')
print(f'CUDA available:   {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU name:         {props.name}')
    print(f'GPU VRAM:         {props.total_memory / 1024**3:.1f} GB')
    print(f'CUDA version:     {torch.version.cuda}')
    print()
    print('✓ GPU is ready. Training will be ~20x faster than CPU.')
else:
    print('✗ No GPU detected.')
    print('  Go to: Runtime → Change runtime type → T4 GPU → Save')
    print('  Then: Runtime → Restart and run all')

## Step 2 — Install dependencies

In [ ]:
# Colab already has torch, numpy, pandas, scikit-learn
# We only need to install the extras
!pip install joblib pyarrow yfinance --quiet
print('Dependencies installed.')

## Step 3 — Upload your project files

Upload these files from your local `apps/ai-trading-service/` folder:
- `model_v2.py`
- `features_v2.py`
- `dataset_v2.py`
- `train_v2.py`
- `utils/trading_v2.py`

OR: just paste the code directly into cells below (faster for experimentation).

In [ ]:
from google.colab import files
import os

print('Upload your .py files from apps/ai-trading-service/')
print('Required: model_v2.py, features_v2.py, dataset_v2.py')
print()
uploaded = files.upload()

# Create utils/ folder and move trading_v2.py into it if uploaded
os.makedirs('utils', exist_ok=True)
if 'trading_v2.py' in uploaded:
    with open('utils/trading_v2.py', 'wb') as f:
        f.write(uploaded['trading_v2.py'])
    print('Moved trading_v2.py → utils/trading_v2.py')

# Create utils/__init__.py
with open('utils/__init__.py', 'w') as f:
    f.write('from .trading_v2 import generate_signal_v2, CONFIDENCE_FLOOR\n')
    f.write('__all__ = ["generate_signal_v2", "CONFIDENCE_FLOOR"]\n')

print('Files uploaded:', list(uploaded.keys()))

## Step 4 — Get training data

**Option A** — Download from Yahoo Finance (free, no API key needed, NSE stocks via .NS suffix)  
**Option B** — Upload your own CSV from Upstox cache

In [ ]:
import yfinance as yf
import pandas as pd

# ── OPTION A: Download from Yahoo Finance ────────────────────────────────────
# NSE symbol format for yfinance: SYMBOL.NS
SYMBOL    = 'RELIANCE'    # Change to your stock
YF_SYMBOL = f'{SYMBOL}.NS'
START     = '2015-01-01'  # Training start date
END       = '2025-01-01'  # Training end date

print(f'Downloading {YF_SYMBOL} from {START} to {END}...')
raw = yf.download(YF_SYMBOL, start=START, end=END, progress=False)

# Rename columns to match our pipeline format
df_raw = raw[['Open','High','Low','Close','Volume']].copy()
df_raw.columns = ['open','high','low','close','volume']
df_raw.index.name = 'datetime'
df_raw = df_raw.reset_index()
df_raw['datetime'] = pd.to_datetime(df_raw['datetime'])
df_raw = df_raw.dropna()

print(f'Downloaded: {len(df_raw):,} rows')
print(f'Date range: {df_raw["datetime"].min().date()} → {df_raw["datetime"].max().date()}')
df_raw.head(3)

In [ ]:
# ── OPTION B: Upload your Upstox CSV / Parquet ───────────────────────────────
# (Run this cell instead of Option A if you have your own data)

# Uncomment and run to upload:
# from google.colab import files
# uploaded = files.upload()
# fname = list(uploaded.keys())[0]
# if fname.endswith('.parquet'):
#     df_raw = pd.read_parquet(fname)
# else:
#     df_raw = pd.read_csv(fname, parse_dates=['datetime'])
# print(f'Loaded: {len(df_raw):,} rows')

## Step 5 — Feature engineering

In [ ]:
from features_v2 import add_features_v2, FEATURE_COLS

df = add_features_v2(df_raw)
print(f'Features computed: {df.shape[1]} columns, {len(df):,} rows')
print(f'Feature columns: {FEATURE_COLS}')

## Step 6 — Create Dataset

In [ ]:
from dataset_v2 import StockDatasetV2

WINDOW           = 60
NOISE_THRESHOLD  = 0.003
VAL_SPLIT        = 0.2

n_total = len(df)
n_val   = int(n_total * VAL_SPLIT)
n_train = n_total - n_val

df_train = df.iloc[:n_train]
df_val   = df.iloc[n_train:]

train_ds = StockDatasetV2(df_train, window=WINDOW, noise_threshold=NOISE_THRESHOLD)
val_ds   = StockDatasetV2(df_val,   window=WINDOW, noise_threshold=NOISE_THRESHOLD,
                          scaler=train_ds.scaler)

train_ds.summary()
val_ds.summary()

## Step 7 — Train the model

In [ ]:
import time
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from model_v2 import StockTransformer

# ── Config ────────────────────────────────────────────────────────────────────
D_MODEL    = 128
N_HEADS    = 8
N_LAYERS   = 4
D_FF       = 256
DROPOUT    = 0.1
BATCH_SIZE = 64
EPOCHS     = 50
LR         = 1e-4
PATIENCE   = 8

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── Weighted sampler ─────────────────────────────────────────────────────────
class_counts   = torch.bincount(train_ds.y_dir)
class_weights  = 1.0 / class_counts.float()
sample_weights = class_weights[train_ds.y_dir]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          pin_memory=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          pin_memory=True, num_workers=2)

# ── Model ─────────────────────────────────────────────────────────────────────
model = StockTransformer(
    input_dim=train_ds.n_features,
    d_model=D_MODEL, n_heads=N_HEADS,
    n_layers=N_LAYERS, d_ff=D_FF, dropout=DROPOUT,
).to(device)

total_p = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_p:,}  |  Size: {total_p*4/1024/1024:.2f} MB')

optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, steps_per_epoch=len(train_loader),
    epochs=EPOCHS, pct_start=0.3)
huber = torch.nn.HuberLoss(delta=0.01)

# ── Training loop ─────────────────────────────────────────────────────────────
print()
print(f'{"Epoch":>5}  {"TrLoss":>7}  {"TrAcc":>6}  {"VaLoss":>7}  {"VaAcc":>6}  {"Time":>6}')
print('-' * 50)

best_val_loss = float('inf')
best_state    = None
patience_ctr  = 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    model.train()
    tr_loss = tr_correct = tr_total = 0
    for X, y_dir, y_ret in train_loader:
        X, y_dir, y_ret = X.to(device), y_dir.to(device), y_ret.to(device)
        logits, ret_p   = model(X)
        loss = F.cross_entropy(logits, y_dir, label_smoothing=0.1) + \
               0.3 * huber(ret_p.squeeze(-1), y_ret)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        tr_loss    += loss.item()
        tr_correct += (logits.argmax(1) == y_dir).sum().item()
        tr_total   += y_dir.size(0)

    model.eval()
    va_loss = va_correct = va_total = 0
    with torch.no_grad():
        for X, y_dir, y_ret in val_loader:
            X, y_dir, y_ret = X.to(device), y_dir.to(device), y_ret.to(device)
            logits, ret_p   = model(X)
            va_loss    += (F.cross_entropy(logits, y_dir, label_smoothing=0.1) +
                           0.3 * huber(ret_p.squeeze(-1), y_ret)).item()
            va_correct += (logits.argmax(1) == y_dir).sum().item()
            va_total   += y_dir.size(0)

    avg_tr = tr_loss / len(train_loader)
    avg_va = va_loss / len(val_loader)
    tr_acc = tr_correct / tr_total
    va_acc = va_correct / va_total

    print(f'{epoch:>5}  {avg_tr:>7.4f}  {tr_acc:>6.3f}  {avg_va:>7.4f}  {va_acc:>6.3f}  {time.time()-t0:>5.1f}s')

    if avg_va < best_val_loss:
        best_val_loss = avg_va
        patience_ctr  = 0
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch} (patience={PATIENCE})')
            break

print(f'\nBest val loss: {best_val_loss:.4f}')

## Step 8 — Save model files

In [ ]:
import joblib

# Save model weights
torch.save(best_state, 'model_v2.pth')
print('Saved: model_v2.pth')

# Save architecture config (needed by infer.py and api_v2.py)
torch.save({
    'input_dim': train_ds.n_features,
    'd_model':   D_MODEL,
    'n_heads':   N_HEADS,
    'n_layers':  N_LAYERS,
    'd_ff':      D_FF,
    'dropout':   DROPOUT,
    'window':    WINDOW,
}, 'model_v2_config.pth')
print('Saved: model_v2_config.pth')

# Save scaler (CRITICAL — must travel with model)
joblib.dump(train_ds.scaler, 'scaler_v2.pkl')
print('Saved: scaler_v2.pkl')

## Step 9 — Download files to your computer

In [ ]:
from google.colab import files

print('Downloading model files to your computer...')
print('Copy these 3 files to your apps/ai-trading-service/ folder.')
print()

files.download('model_v2.pth')
files.download('model_v2_config.pth')
files.download('scaler_v2.pkl')

print('Done! Now run: python infer.py --symbol RELIANCE')

## Step 10 — (Optional) Save to Google Drive

Instead of downloading, you can save to Google Drive so you never lose the trained model.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil
# dest = '/content/drive/MyDrive/ai-trading-models/'
# os.makedirs(dest, exist_ok=True)
# shutil.copy('model_v2.pth', dest)
# shutil.copy('model_v2_config.pth', dest)
# shutil.copy('scaler_v2.pkl', dest)
# print(f'Saved to Google Drive: {dest}')

print('Uncomment the lines above to save to Google Drive.')